# A1 · Logistic regression with gradient descent

**Type**: Implement from scratch · **Level**: ●●○ independent · **Time**: about 50 minutes · **core path ★**

## Objective

Implement with numpy an L2-regularized logistic regression (loss function, gradient, gradient descent), check the gradient with finite
differences and show that the coefficients match those of `LogisticRegression`. Then show **why** without regularization the method does not converge
on nearly separable data.

## Dataset

`load_breast_cancer` (569 nodules, 30 measurements, target 1 = benign) split 70/30 and standardized: `Z_tr, y_tr, Z_te, y_te`.

## Educational purpose

Understand what `fit` does: an iterative solver that minimizes a convex log-loss. Above all: understand the **relationship between `C` and the penalty** (`C` multiplies the
log-loss, not the penalty) and see perfect separation in action, which you will need in exercise C1.

## Task

1. Write `sigmoid(z)`, `loss(w, b, Z, y, C)` and `gradient(w, b, Z, y, C)` (returns `(grad_w, grad_b)`). The scikit-learn objective $C\sum_i \ell_i + \tfrac12\lVert w\rVert^2$ divided by $C n$ is the mean loss $\frac1n\sum_i \ell_i + \frac{1}{2Cn}\lVert w\rVert^2$ (the intercept is **not** penalized).
2. Check the gradient with finite differences: save in `gradient_error` the maximum absolute gap between analytic and numeric gradient (step `1e-6`, central difference) over all the components of `w` and over `b`, computed at a random point `w0` (given).
3. Write `train(Z, y, C, lr, n_iter)`: gradient descent from `w = 0, b = 0`; return `w, b, history` (the loss at every iteration).
4. Train with `C=1, lr=0.5, n_iter=5000` and compare with `LogisticRegression(C=1, tol=1e-10, max_iter=100000)`: save `coef_gap` (maximum absolute gap on the coefficients).
5. **Without regularization:** train with `C=1e6` for 500 and for 5000 iterations and save the Euclidean norms of the weights in `norms = (norm_500, norm_5000)`. The weights keep growing: why?

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.3, stratify=data.target, random_state=0)
scaler = StandardScaler().fit(X_tr)
Z_tr, Z_te = scaler.transform(X_tr), scaler.transform(X_te)
w0 = np.random.default_rng(0).normal(0, 0.3, Z_tr.shape[1])         # point at which to check the gradient

## Your work

### 1. Loss and gradient

In [ ]:
def sigmoid(z):
    ...


def loss(w, b, Z, y, C):
    ...


def gradient(w, b, Z, y, C):
    ...

### 2. Check with finite differences

In [ ]:
gradient_error = None

### 3. Gradient descent

In [ ]:
def train(Z, y, C, lr, n_iter):
    ...

In [ ]:
w, b, history = train(Z_tr, y_tr, C=1, lr=0.5, n_iter=5000)
sk = LogisticRegression(C=1, tol=1e-10, max_iter=100000).fit(Z_tr, y_tr)
coef_gap = None

### 4. Without regularization

In [ ]:
norms = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`p = sigmoid(Z @ w + b)`; `grad_w = Z.T @ (p - y) / n + w / (C * n)`; `grad_b = (p - y).mean()`. Stable sigmoid: `1 / (1 + np.exp(-np.clip(z, -500, 500)))`. Finite differences: for every component `(loss(w + h e_j) - loss(w - h e_j)) / (2h)`.'):
    check(close(sigmoid(np.array([-50.0, 0.0, 50.0])), [0.0, 0.5, 1.0], atol=1e-12), "sigmoid correct at the extremes (and without overflow)", "sigmoid(-50) = 0, sigmoid(0) = 0.5, sigmoid(50) = 1")
    check(close(loss(np.zeros(30), 0.0, Z_tr, y_tr, 1.0), np.log(2)), "with zero weights the loss is ln 2", "with w = 0 and b = 0 every example costs ln(2)")
    check(gradient_error is not None and gradient_error < 1e-6, "analytic gradient equal to the numeric one (gap < 1e-6)", "check the formula: Z.T @ (p - y) / n + w / (C n) for w, and the mean of (p - y) for b")
    check(len(history) == 5000 and all(np.diff(history) <= 1e-12), "the loss never goes up", "with a small enough step the loss must go down at every iteration")
    check(coef_gap is not None and coef_gap < 1e-3, "the coefficients match those of scikit-learn (gap < 1e-3)", "did you penalize the intercept? used 1/C instead of 1/(C n)?")
    check(norms is not None and norms[1] > 1.5 * norms[0] and norms[1] > 5, "without regularization the norm of the weights keeps growing", "norm of the weights after 5000 iterations against 500, with C = 1e6")
    check(np.all(((Z_te @ w + b) > 0).astype(int) == sk.predict(Z_te)), "the predictions on the test set match those of scikit-learn", "same rule: class 1 if z > 0")
done()

---
The commented solution is in `solutions/A_from_scratch/A1_logistic_with_gradient_descent.ipynb`. Open it after you pass the check, or after you have really tried.